# Inhomogeneous cross-section eigensolver: dielectric-loaded WR-90Drive `yee.eigensolver.NumericalCrossSection.solve` on a WR-90 rectangular waveguide whose **lower half is filled with FR-4** ($\varepsilon_r = 4.4$) while the upper half stays air. Unlike the homogeneous air-filled case (`eigensolver_wr90_te10.ipynb`), the dominant mode here sees an **inhomogeneous** cross-section: the dielectric slows the wave, raising the propagation constant $\beta$ and the effective permittivity $\varepsilon_\mathrm{eff} = (\beta^2 + (\pi/a)^2)/k_0^2$, and lowering the wave impedance $Z_w$.We solve an **air** guide and the **FR-4-loaded** guide at 10 GHz and compare. The loaded $\beta$ must land strictly inside the rigorous monotonic bracket $\beta_\mathrm{air} < \beta_\mathrm{loaded} < \beta_\mathrm{fully\,filled}$ — the same physics the Rust `fr4_loaded_beta_matches_reference` gate checks.

In [ ]:
import math

import yee
# Both import styles work — `yee.eigensolver` is registered in `sys.modules`
# by the PyO3 init (mirrors the `yee.touchstone` pattern).
from yee.eigensolver import NumericalCrossSection, TriMesh2D

A = 22.86e-3  # WR-90 long inner dimension (m)
B = 10.16e-3  # WR-90 short inner dimension (m)
FREQ_HZ = 10.0e9
EPS_FR4 = 4.4  # FR-4 relative permittivity
C0 = 299_792_458.0

## Build the cross-section meshesA structured `nx \u00d7 ny` quad grid spanning `[0, a] \u00d7 [0, b]`, each quad split along its lower-left \u2192 upper-right diagonal into two CCW triangles (the `TriMesh2D` constructor enforces strictly positive signed area). For the loaded guide, each triangle is tagged by its quad's `y`-midpoint: tag `1` (FR-4) below `y = b/2`, tag `0` (air) above. The air guide leaves every triangle on tag `0`.

In [ ]:
def slab_mesh(nx, ny, loaded):
    """WR-90 quad-grid mesh. If `loaded`, the lower-y half is tag 1."""
    vertices = [
        (A * i / nx, B * j / ny)
        for j in range(ny + 1)
        for i in range(nx + 1)
    ]

    def idx(i, j):
        return j * (nx + 1) + i

    triangles = []
    triangle_material = []
    for j in range(ny):
        y_mid = B * (j + 0.5) / ny
        tag = 1 if (loaded and y_mid < B / 2.0) else 0
        for i in range(nx):
            v00 = idx(i, j)
            v10 = idx(i + 1, j)
            v11 = idx(i + 1, j + 1)
            v01 = idx(i, j + 1)
            triangles.append((v00, v10, v11))
            triangle_material.append(tag)
            triangles.append((v00, v11, v01))
            triangle_material.append(tag)
    return TriMesh2D(vertices, triangles, triangle_material=triangle_material)


NX, NY = 8, 8  # 128 triangles, 81 vertices
air = slab_mesh(NX, NY, loaded=False)
loaded = slab_mesh(NX, NY, loaded=True)
print(f"mesh: n_verts = {loaded.n_verts()}, n_tris = {loaded.n_tris()}")

## Solve and compare`eps_r` / `mu_r` are `dict[int, complex]` keyed by material tag. The air guide maps both tags to $1 + 0j$; the loaded guide maps tag `1` to FR-4. After `solve`, read back `nc.beta`, `nc.z_w`, and the per-vertex longitudinal field `nc.mode_profile_ez`, and form $\varepsilon_\mathrm{eff}$ from $\beta$.

In [ ]:
def eps_eff(beta_real):
    k0 = 2.0 * math.pi * FREQ_HZ / C0
    kc = math.pi / A
    return (beta_real**2 + kc**2) / k0**2


def solve_guide(mesh, eps_tag1):
    nc = NumericalCrossSection(
        mesh,
        eps_r={0: complex(1.0, 0.0), 1: complex(eps_tag1, 0.0)},
        mu_r={0: complex(1.0, 0.0), 1: complex(1.0, 0.0)},
    )
    nc.solve(FREQ_HZ)
    return nc


nc_air = solve_guide(air, 1.0)
nc_loaded = solve_guide(loaded, EPS_FR4)

for label, nc in (("air", nc_air), ("FR-4 loaded", nc_loaded)):
    beta = nc.beta.real  # lossless => Im ~ 0
    zw = nc.z_w.real
    print(
        f"{label:12s}: \u03b2 = {beta:8.3f} rad/m | "
        f"\u03b5_eff = {eps_eff(beta):6.3f} | Z_w = {zw:7.2f} \u03a9"
    )

## Verify the monotonic bracketWith $k_c = \pi/a$ fixed by the PEC walls, the empty and fully-filled TE10 closed forms $\beta = \sqrt{\varepsilon_r k_0^2 - (\pi/a)^2}$ bracket any partial fill. The half-filled $\beta$ must sit strictly between them, and its $\varepsilon_\mathrm{eff}$ between $1$ and $\varepsilon_r$.

In [ ]:
def te10_beta(eps_r):
    k = 2.0 * math.pi * FREQ_HZ * math.sqrt(eps_r) / C0
    kc = math.pi / A
    return math.sqrt(k * k - kc * kc)


beta_air = te10_beta(1.0)
beta_full = te10_beta(EPS_FR4)
beta_loaded = nc_loaded.beta.real
print(f"\u03b2_air         = {beta_air:8.3f} rad/m")
print(f"\u03b2_loaded      = {beta_loaded:8.3f} rad/m  (half FR-4)")
print(f"\u03b2_fully-filled = {beta_full:8.3f} rad/m")
assert beta_air < beta_loaded < beta_full, "bracket violated!"
print(
    f"\nbracket OK: dielectric loading raised \u03b2 by "
    f"{(beta_loaded / beta_air - 1.0) * 100.0:.1f} % over air, "
    f"staying below the fully-filled limit."
)

## Longitudinal field $E_z$On an air-filled guide the dominant mode is purely transverse, so `mode_profile_ez` is $\approx 0$. The dielectric interface makes the loaded mode genuinely hybrid, so a non-zero longitudinal field appears. We compare $\lVert E_z \rVert$ (one complex amplitude per mesh vertex) for the two guides.

In [ ]:
def ez_norm(nc):
    ez = nc.mode_profile_ez  # list[complex], one per global vertex
    return math.sqrt(sum(abs(c) ** 2 for c in ez))


print(f"len(mode_profile_ez) = {len(nc_loaded.mode_profile_ez)} "
      f"(== n_verts = {loaded.n_verts()})")
print(f"\u2016E_z\u2016 air         = {ez_norm(nc_air):.4e}")
print(f"\u2016E_z\u2016 FR-4 loaded = {ez_norm(nc_loaded):.4e}")

The FR-4-loaded guide lands $\beta \approx 329$ rad/m (modal $\varepsilon_\mathrm{eff} \approx 2.9$, between air and the FR-4 fill) on this 8×8 mesh — well inside the air/full bracket and within ~1.3 % of the published LSM-to-y transverse-resonance reference (see `crates/yee-mom/tests/eigensolver_inhomogeneous.rs`). Refining the mesh or raising the contrast (e.g. $\varepsilon_r = 10.2$) is straightforward; at high contrast the first-order elements under-resolve the interface field peak, which is the documented step-5.4 higher-order-element item.